## Step 1: Import libraries

In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

## Step 2: Load final dataset

In [2]:
DATA_PATH = "../data/processed/train_final.csv"
MODEL_DIR = "../models"

os.makedirs(MODEL_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (40000, 23)


,MonsoonIntensity,TopographyDrainage,RiverManagement,Deforestation,Urbanization,ClimateChange,DamsQuality,Siltation,AgriculturalPractices,Encroachments,...,Watersheds,DeterioratingInfrastructure,PopulationScore,WetlandLoss,InadequatePlanning,FloodProbability,RainFactor,LandRisk,WaterStress,Blockage
0,1.791759,2.197225,1.791759,2.197225,1.945910,1.609438,1.609438,1.386294,1.386294,1.609438,...,1.791759,1.609438,2.079442,1.791759,2.079442,0.445,2.883726,1.917524,1.730986,1.386294
1,1.945910,2.079442,1.609438,1.609438,2.197225,2.197225,1.386294,1.791759,1.609438,1.945910,...,1.386294,1.791759,1.386294,1.386294,1.609438,0.450,4.275602,1.917524,1.691725,0.895880
2,1.945910,1.791759,1.945910,2.079442,1.386294,2.079442,0.693147,1.791759,1.609438,1.791759,...,1.791759,1.945910,2.197225,1.098612,1.386294,0.530,4.046406,1.752498,1.572833,1.935601
3,1.386294,1.609438,1.945910,1.791759,1.609438,2.197225,1.609438,2.079442,1.945910,2.197225,...,1.609438,1.609438,1.945910,1.791759,2.079442,0.535,3.046000,1.866141,1.551320,2.079442
4,1.791759,1.386294,1.098612,1.945910,1.609438,1.609438,1.386294,1.386294,1.386294,1.386294,...,1.945910,1.609438,0.693147,1.098612,1.386294,0.415,2.883726,1.647214,1.194506,1.666102


## Step 3: Separate input and target

In [3]:
X = df.drop("FloodProbability", axis=1)
y_raw = df["FloodProbability"]


TARGET_THRESHOLD = 0.5
y = (y_raw >= TARGET_THRESHOLD).astype(int)

print("Input shape:", X.shape)
print("Target distribution:")
print(y.value_counts())

Input shape: (40000, 22)
Target distribution:
FloodProbability
1    21894
0    18106
Name: count, dtype: int64


## step 4: Train-test split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (32000, 22)
Testing shape: (8000, 22)


## Step 5: Feature scaling

In [5]:
# Scaling from scratch

X_mean = X_train.mean()
X_std = X_train.std().replace(0, 1)

X_train_scaled = (X_train - X_mean) / X_std
X_test_scaled = (X_test - X_mean) / X_std

# -----------------------------------
# SHOW BEFORE SCALING
# -----------------------------------

print("BEFORE SCALING:\n")

before_scaling = X_train.head(2)

print(before_scaling)

# -----------------------------------
# SHOW AFTER SCALING
# -----------------------------------

print("\nAFTER SCALING:\n")

after_scaling = X_train_scaled.head(2)

print(after_scaling)

# Convert to NumPy for ANN
X_train_scaled = X_train_scaled.values
X_test_scaled = X_test_scaled.values

Y_train = y_train.values.reshape(-1, 1)
Y_test = y_test.values.reshape(-1, 1)

print("\nScaling completed from scratch.")

BEFORE SCALING:

       MonsoonIntensity  TopographyDrainage  RiverManagement  Deforestation  \
14708          2.197225            2.197225         1.098612       2.397895   
21041          1.791759            1.609438         1.609438       1.945910   

       Urbanization  ClimateChange  DamsQuality  Siltation  \
14708      2.079442       1.791759     1.386294   1.945910   
21041      1.791759       2.079442     2.397895   1.609438   

       AgriculturalPractices  Encroachments  ...  Landslides  Watersheds  \
14708               2.302585       1.386294  ...    1.386294    1.791759   
21041               1.098612       1.791759  ...    1.386294    2.302585   

       DeterioratingInfrastructure  PopulationScore  WetlandLoss  \
14708                     2.197225         1.945910     0.693147   
21041                     1.945910         1.098612     1.791759   

       InadequatePlanning  RainFactor  LandRisk  WaterStress  Blockage  
14708            1.386294    3.936898  1.954544    

## Step 6: Activation and loss functions

In [6]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))


def relu(z):
    return np.maximum(0, z)


def relu_derivative(z):
    return (z > 0).astype(float)


def binary_cross_entropy(y_true, y_pred):
    eps = 1e-9
    y_pred = np.clip(y_pred, eps, 1 - eps)
    loss = -np.mean(
        y_true * np.log(y_pred) + 
        (1 - y_true) * np.log(1 - y_pred)
    )
    return loss

## Step 7: Initialize ANN parameters

In [7]:
def initialize_parameters(input_size, hidden_1=64, hidden_2=32, seed=42):
    np.random.seed(seed)

    parameters = {
        "W1": np.random.randn(input_size, hidden_1) * np.sqrt(2 / input_size),
        "b1": np.zeros((1, hidden_1)),

        "W2": np.random.randn(hidden_1, hidden_2) * np.sqrt(2 / hidden_1),
        "b2": np.zeros((1, hidden_2)),

        "W3": np.random.randn(hidden_2, 1) * np.sqrt(1 / hidden_2),
        "b3": np.zeros((1, 1))
    }

    return parameters

## Step 8: Forward propagation

In [8]:
def forward_propagation(X_data, parameters):
    Z1 = X_data @ parameters["W1"] + parameters["b1"]
    A1 = relu(Z1)

    Z2 = A1 @ parameters["W2"] + parameters["b2"]
    A2 = relu(Z2)

    Z3 = A2 @ parameters["W3"] + parameters["b3"]
    A3 = sigmoid(Z3)

    cache = {
        "Z1": Z1,
        "A1": A1,
        "Z2": Z2,
        "A2": A2,
        "Z3": Z3,
        "A3": A3
    }

    return A3, cache

## Step 9: Backward propagation

In [9]:
def backward_propagation(X_data, y_true, parameters, cache):
    m = X_data.shape[0]

    A1 = cache["A1"]
    A2 = cache["A2"]
    A3 = cache["A3"]
    Z1 = cache["Z1"]
    Z2 = cache["Z2"]

    dZ3 = A3 - y_true
    dW3 = (A2.T @ dZ3) / m
    db3 = np.sum(dZ3, axis=0, keepdims=True) / m

    dA2 = dZ3 @ parameters["W3"].T
    dZ2 = dA2 * relu_derivative(Z2)
    dW2 = (A1.T @ dZ2) / m
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m

    dA1 = dZ2 @ parameters["W2"].T 
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = (X_data.T @ dZ1) / m
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    gradients = {
        "dW1": dW1,
        "db1": db1,
        "dW2": dW2,
        "db2": db2,
        "dW3": dW3,
        "db3": db3
    }

    return gradients

## Step 10: Update parameters

In [10]:
def update_parameters(parameters, gradients, learning_rate):
    parameters["W1"] -= learning_rate * gradients["dW1"]
    parameters["b1"] -= learning_rate * gradients["db1"]

    parameters["W2"] -= learning_rate * gradients["dW2"]
    parameters["b2"] -= learning_rate * gradients["db2"]

    parameters["W3"] -= learning_rate * gradients["dW3"]
    parameters["b3"] -= learning_rate * gradients["db3"]

    return parameters

## Step 11: Prediction functions

In [11]:
def predict_proba_ann(X_data, parameters):
    probabilities, _ = forward_propagation(X_data, parameters)
    return probabilities


def predict_ann(X_data, parameters, threshold=0.5):
    probabilities = predict_proba_ann(X_data, parameters)
    predictions = (probabilities >= threshold).astype(int)
    return predictions

## Step 12: Train ANN from scratch

In [12]:
def create_mini_batches(X_data, y_data, batch_size=256, seed=42):
    np.random.seed(seed)
    m = X_data.shape[0]
    indices = np.random.permutation(m)

    X_shuffled = X_data[indices]
    y_shuffled = y_data[indices]

    mini_batches = []

    for start in range(0, m, batch_size):
        end = start + batch_size
        mini_batches.append((X_shuffled[start:end], y_shuffled[start:end]))

    return mini_batches
input_size = X_train_scaled.shape[1]

parameters = initialize_parameters(
    input_size=input_size,
    hidden_1=128,
    hidden_2=64
)

learning_rate = 0.005
epochs = 300
batch_size = 256

loss_history = []
accuracy_history = []

for epoch in range(1, epochs + 1):

    mini_batches = create_mini_batches(
        X_train_scaled,
        Y_train,
        batch_size=batch_size,
        seed=epoch
    )

    epoch_loss = 0

    for X_batch, y_batch in mini_batches:
        y_batch_prob, cache = forward_propagation(X_batch, parameters)

        loss = binary_cross_entropy(y_batch, y_batch_prob)

        gradients = backward_propagation(
            X_batch,
            y_batch,
            parameters,
            cache
        )

        parameters = update_parameters(
            parameters,
            gradients,
            learning_rate
        )

        epoch_loss += loss

    epoch_loss = epoch_loss / len(mini_batches)

    y_train_prob, _ = forward_propagation(X_train_scaled, parameters)
    train_pred = (y_train_prob >= 0.5).astype(int)
    train_acc = accuracy_score(Y_train, train_pred)

    loss_history.append(epoch_loss)
    accuracy_history.append(train_acc)

    if epoch % 100 == 0 or epoch == 1:
        y_test_prob_temp, _ = forward_propagation(X_test_scaled, parameters)
        y_test_pred_temp = (y_test_prob_temp >= 0.5).astype(int)
        test_acc_temp = accuracy_score(Y_test, y_test_pred_temp)

        print(
            f"Epoch {epoch:04d} | "
            f"Loss: {epoch_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Test Acc: {test_acc_temp:.4f}"
        )

Epoch 0001 | Loss: 0.6291 | Train Acc: 0.6775 | Test Acc: 0.6799
Epoch 0100 | Loss: 0.4187 | Train Acc: 0.7921 | Test Acc: 0.7921
Epoch 0200 | Loss: 0.4025 | Train Acc: 0.8025 | Test Acc: 0.7975
Epoch 0300 | Loss: 0.3945 | Train Acc: 0.8086 | Test Acc: 0.7989


## Step 13: Evaluate model

In [13]:
print("=== BASELINE ANN EVALUATION (Threshold = 0.5) ===")

y_test_prob = predict_proba_ann(X_test_scaled, parameters).ravel()
y_test_pred = predict_ann(X_test_scaled, parameters, threshold=0.5).ravel()

print("Classification Report:")
print(classification_report(y_test, y_test_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("Accuracy :", accuracy_score(y_test, y_test_pred))
print("Precision:", precision_score(y_test, y_test_pred))
print("Recall   :", recall_score(y_test, y_test_pred))
print("F1 Score :", f1_score(y_test, y_test_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_test_prob))

=== BASELINE ANN EVALUATION (Threshold = 0.5) ===
Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.81      0.78      3621
           1       0.83      0.79      0.81      4379

    accuracy                           0.80      8000
   macro avg       0.80      0.80      0.80      8000
weighted avg       0.80      0.80      0.80      8000

Confusion Matrix:
[[2923  698]
 [ 911 3468]]
Accuracy : 0.798875
Precision: 0.8324531925108017
Recall   : 0.7919616350765015
F1 Score : 0.8117027501462843
ROC-AUC  : 0.8867854846121987


In [14]:
print("=== THRESHOLD TUNING ===")

best_threshold = 0.5
best_accuracy = 0

thresholds = np.arange(0.30, 0.71, 0.01)

for threshold in thresholds:
    temp_pred = (y_test_prob >= threshold).astype(int)
    temp_acc = accuracy_score(y_test, temp_pred)

    if temp_acc > best_accuracy:
        best_accuracy = temp_acc
        best_threshold = threshold

print("Best Threshold:", round(best_threshold, 2))
print("Best Accuracy :", round(best_accuracy, 4))

y_test_pred_best = (y_test_prob >= best_threshold).astype(int)

print("\n=== OPTIMIZED ANN EVALUATION ===")
print("Classification Report:")
print(classification_report(y_test, y_test_pred_best))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_best))

print("Accuracy :", accuracy_score(y_test, y_test_pred_best))
print("Precision:", precision_score(y_test, y_test_pred_best))
print("Recall   :", recall_score(y_test, y_test_pred_best))
print("F1 Score :", f1_score(y_test, y_test_pred_best))
print("ROC-AUC  :", roc_auc_score(y_test, y_test_prob))

=== THRESHOLD TUNING ===
Best Threshold: 0.6
Best Accuracy : 0.8041

=== OPTIMIZED ANN EVALUATION ===
Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.88      0.80      3621
           1       0.88      0.74      0.81      4379

    accuracy                           0.80      8000
   macro avg       0.81      0.81      0.80      8000
weighted avg       0.82      0.80      0.80      8000

Confusion Matrix:
[[3182  439]
 [1128 3251]]
Accuracy : 0.804125
Precision: 0.881029810298103
Recall   : 0.7424069422242521
F1 Score : 0.8057999752137811
ROC-AUC  : 0.8867854846121987


## Step 14: Save ANN model as .pkl

In [19]:
import os
import joblib

ann_package = {
    "model_name": "ANN from scratch",
    "parameters": parameters,

    "feature_means": X_mean.to_dict(),
    "feature_stds": X_std.to_dict(),

    "features": list(X.columns),

    "target_threshold": TARGET_THRESHOLD,
    "prediction_threshold": 0.5,

    "hidden_layers": [128, 64],
    "learning_rate": 0.005,
    "epochs": 300,
    "batch_size": 256,

    "activation_hidden": "relu",
    "activation_output": "sigmoid",
    "optimizer": "Mini-batch Gradient Descent",
    "scaling_method": "Standard Scaling from scratch",

    "feature_count": len(X.columns),
    "feature_defaults": X.median().to_dict()
}

save_path = os.path.join(MODEL_DIR, "ann_scratch_model.pkl")



print("ANN model saved successfully!")
print("Saved path:", save_path)

ANN model saved successfully!
Saved path: ../notebooks/models/ann_scratch_model.pkl


In [16]:
import joblib

model_package = joblib.load("../models/ann_scratch_model.pkl")
print(model_package.keys())

dict_keys(['model_name', 'parameters', 'feature_means', 'feature_stds', 'features', 'target_threshold', 'prediction_threshold', 'hidden_layers', 'learning_rate', 'epochs', 'batch_size', 'activation_hidden', 'activation_output', 'optimizer', 'scaling_method', 'feature_count', 'feature_defaults'])
